# Experiment: Private To Public Case Extraction Gate

Objective:
- Model the private-source to public-summary gate for case-linked facts.
- Keep raw records, identifiers, local paths, and treatment instructions blocked.
- Produce only de-identified extraction decisions that can be checked before public commit.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Final

RELEASE_ORDER: Final = (
    "private_raw_record",
    "public_blocked_identifier",
    "public_blocked_treatment_claim",
    "deidentified_extract_candidate",
    "source_link_only",
    "public_summary_ready",
)

RELEASE_ORDER

## Gate Model

This is not a legal de-identification engine. It is a repo-release model for deciding whether a private-source fact can become public case context.


In [ ]:
@dataclass(frozen=True)
class ExtractionSignal:
    """A privacy or claim signal in a candidate extraction."""

    name: str
    release_class: str
    present: bool
    reason: str


def release_decision(signals: list[ExtractionSignal]) -> dict[str, object]:
    """Return the strongest release decision for a candidate extraction."""
    present_signals = [signal for signal in signals if signal.present]

    if not present_signals:
        return {
            "decision": "public_summary_ready",
            "blockers": [],
            "review_notes": [],
        }

    ordered = sorted(
        present_signals,
        key=lambda signal: RELEASE_ORDER.index(signal.release_class),
    )
    decision = ordered[0].release_class
    blockers = [
        signal.name
        for signal in ordered
        if signal.release_class.startswith("public_blocked")
        or signal.release_class == "private_raw_record"
    ]

    return {
        "decision": decision,
        "blockers": blockers,
        "review_notes": [signal.reason for signal in ordered],
    }

## Scenario Smoke Test

The scenarios mirror the mistakes the public repository must prevent before a case fact is committed.


In [ ]:
scenarios = {
    "raw_pdf_with_local_path": [
        ExtractionSignal(
            name="raw_pdf",
            release_class="private_raw_record",
            present=True,
            reason="Raw medical PDFs stay outside Git.",
        ),
        ExtractionSignal(
            name="local_source_path",
            release_class="private_raw_record",
            present=True,
            reason="Local source paths are private working context.",
        ),
    ],
    "direct_identifier": [
        ExtractionSignal(
            name="record_identifier",
            release_class="public_blocked_identifier",
            present=True,
            reason="Record identifiers cannot be public.",
        )
    ],
    "therapy_action_claim": [
        ExtractionSignal(
            name="therapy_action",
            release_class="public_blocked_treatment_claim",
            present=True,
            reason="Treatment actions require clinician review.",
        )
    ],
    "deidentified_question_packet": [],
}

scenario_results = {
    name: release_decision(signals) for name, signals in scenarios.items()
}

assert scenario_results["raw_pdf_with_local_path"]["decision"] == ("private_raw_record")
assert scenario_results["direct_identifier"]["decision"] == (
    "public_blocked_identifier"
)
assert scenario_results["therapy_action_claim"]["decision"] == (
    "public_blocked_treatment_claim"
)
assert scenario_results["deidentified_question_packet"]["decision"] == (
    "public_summary_ready"
)

scenario_results

## Minimal Fact Extraction Shape

A public case fact should carry a source class, release class, evidence label, and a reason why the fact matters. It should not carry the raw document or a direct identifier.


In [ ]:
@dataclass(frozen=True)
class PublicFactCandidate:
    """A de-identified fact candidate for public case context."""

    field: str
    value_class: str
    evidence_label: str
    why_it_matters: str
    release_class: str

    def is_ready(self) -> bool:
        """Return whether the candidate can be public after review."""
        return self.release_class == "public_summary_ready"


candidate = PublicFactCandidate(
    field="historical_broad_timing",
    value_class="infancy timing only",
    evidence_label="extracted historical context",
    why_it_matters="routes genotype and HbF questions without identifiers",
    release_class="public_summary_ready",
)

assert candidate.is_ready()
candidate

## Result

The durable rule is simple: raw source files stay private; public text carries only the minimum de-identified fact, uncertainty label, and clinician-review boundary.
